In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE INSURANCE_PROJECT.MARTS.CLAIMS_SUMMARY AS

WITH claims_summarized AS (
    SELECT
       claim_id,
       claim_status,
       COUNT(*) AS claim_count,
       AVG(claim_amount) AS avg_claims,
       SUM(claim_amount) AS total_claim_amount,
       AVG(fraud_score) AS avg_fraud_score
    FROM INSURANCE_PROJECT.MARTS.CLAIMS_INFORMATION
    GROUP BY claim_id, claim_status
)

SELECT *
FROM claims_summarized;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE FUNCTION INSURANCE_PROJECT.MARTS.fraud_risk_category(
    fraud_score NUMBER(18, 1)
)
RETURNS VARCHAR(100)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.12'
HANDLER = 'get_fraud_risk'
AS
$$
def get_fraud_risk(fraud_score):
    if fraud_score is None:
        return 'Unknown'
    elif fraud_score >= 9.5:
        return 'High'
    elif fraud_score >= 8.0:
        return 'Medium'
    else:
        return 'Low'
$$;


In [ ]:
%%sql -r dataframe_7
SELECT
    fraud_score,
    INSURANCE_PROJECT.MARTS.fraud_risk_category(fraud_score) AS fraud_risk_category
FROM INSURANCE_PROJECT.CORE.FACT_CLAIM
LIMIT 20;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TABLE INSURANCE_PROJECT.MARTS.POLICIES_SUMMARY AS

WITH aggregated_claims AS (
    SELECT
        policy_key,
        SUM(COALESCE(claim_amount, 0)) AS total_claim_amount,
        COUNT(claim_id) AS claim_count
    FROM INSURANCE_PROJECT.MARTS.CLAIMS_INFORMATION
    GROUP BY policy_key
),

FINAL_SELECTED_COLUMNS AS (
    SELECT
        p.policy_key,
        p.policy_id,
        p.policy_type,
        p.policy_status,
        p.annual_premium,
        p.deductible,
        COALESCE(c.claim_count, 0) AS claim_count,
        COALESCE(c.total_claim_amount, 0) AS total_claim_amount,
        COALESCE(c.total_claim_amount, 0)
            / NULLIF(p.annual_premium, 0) AS claim_to_premium_ratio
    FROM INSURANCE_PROJECT.MARTS.POLICY_INFORMATION p
    LEFT JOIN aggregated_claims c
        ON p.policy_key = c.policy_key
)

SELECT *
FROM FINAL_SELECTED_COLUMNS;

In [ ]:
%%sql -r dataframe_1
SELECT COUNT(*) AS matching_claims
FROM INSURANCE_PROJECT.CORE.DIM_POLICY p
JOIN INSURANCE_PROJECT.CORE.FACT_CLAIM c
    ON p.policy_key = c.policy_key;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE INSURANCE_PROJECT.MARTS.CUSTOMERS_SUMMARY AS

WITH customers_selected AS (
    SELECT
        customer_key,
        customer_id,
        occupation,
        city,
        region
    FROM INSURANCE_PROJECT.MARTS.CUSTOMER_INFORMATION
),

policy_aggregated AS (
    SELECT
        customer_key,
        COUNT(DISTINCT policy_id) AS policy_count
    FROM INSURANCE_PROJECT.MARTS.POLICY_INFORMATION
    GROUP BY customer_key
),

claims_aggregated AS (
    SELECT
        p.customer_key,
        COUNT(c.claim_id) AS claim_count,
        SUM(c.claim_amount) AS total_claim_amount,
        AVG(c.claim_amount) AS avg_claim_amount,
        AVG(c.fraud_score) AS avg_fraud_score
    FROM INSURANCE_PROJECT.MARTS.CLAIMS_INFORMATION c
    INNER JOIN INSURANCE_PROJECT.MARTS.POLICY_INFORMATION p
        ON c.policy_key = p.policy_key
    GROUP BY p.customer_key
),

payments_aggregated AS (
    SELECT
        fc.customer_key,
        SUM(fp.payment_amount) AS total_paid,
        COUNT(fp.payment_id) AS payment_count
    FROM INSURANCE_PROJECT.CORE.FACT_PAYMENT fp
    INNER JOIN INSURANCE_PROJECT.CORE.FACT_CLAIM fc
        ON fp.claim_key = fc.claim_key
    GROUP BY fc.customer_key
),

final_selected_columns AS (
    SELECT
        c.customer_key,
        c.customer_id,
        c.occupation,
        c.city,
        c.region,

        COALESCE(p.policy_count, 0) AS policy_count,

        COALESCE(cl.claim_count, 0) AS claim_count,
        COALESCE(cl.total_claim_amount, 0) AS total_claim_amount,
        COALESCE(cl.avg_claim_amount, 0) AS avg_claim_amount,
        COALESCE(cl.avg_fraud_score, 0) AS avg_fraud_score,

        COALESCE(pay.total_paid, 0) AS total_paid,
        COALESCE(pay.payment_count, 0) AS payment_count

    FROM customers_selected c

    LEFT JOIN policy_aggregated p
        ON c.customer_key = p.customer_key

    LEFT JOIN claims_aggregated cl
        ON c.customer_key = cl.customer_key

    LEFT JOIN payments_aggregated pay
        ON c.customer_key = pay.customer_key
)

SELECT *
FROM final_selected_columns;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE INSURANCE_PROJECT.MARTS.MONTHLY_CLAIMS_SUMMARY AS

WITH claims_monthly AS (
    SELECT
        d.year,
        d.month,
        d.month_name,
        COUNT(fc.claim_id) AS claim_count,
        SUM(fc.claim_amount) AS total_claim_amount,
        AVG(fc.claim_amount) AS avg_claim_amount,
        AVG(fc.fraud_score) AS avg_fraud_score
    FROM INSURANCE_PROJECT.CORE.FACT_CLAIM fc
    INNER JOIN INSURANCE_PROJECT.CORE.DIM_DATES d
        ON fc.date_key = d.date_key
    GROUP BY
        d.year,
        d.month,
        d.month_name
),

payments_monthly AS (
    SELECT
        d.year,
        d.month,
        SUM(fp.payment_amount) AS total_paid,
        COUNT(fp.payment_id) AS payment_count
    FROM INSURANCE_PROJECT.CORE.FACT_PAYMENT fp
    INNER JOIN INSURANCE_PROJECT.CORE.DIM_DATES d
        ON fp.date_key = d.date_key
    GROUP BY
        d.year,
        d.month
),

final_selected_columns AS (
    SELECT
        c.year,
        c.month,
        c.month_name,
        c.claim_count,
        c.total_claim_amount,
        c.avg_claim_amount,
        c.avg_fraud_score,
        COALESCE(p.total_paid, 0) AS total_paid,
        COALESCE(p.payment_count, 0) AS payment_count
    FROM claims_monthly c
    LEFT JOIN payments_monthly p
        ON c.year = p.year
       AND c.month = p.month
)

SELECT *
FROM final_selected_columns;